# Model comparison and results

This notebook compares the performance and size of all CIFAR-10 models trained in previous notebooks:
- **01-DNN**: Fully connected deep neural network
- **02-CNN**: Convolutional neural network (grayscale)
- **03-RGB-CNN**: Convolutional neural network (RGB)
- **04-architecture_optimization**: Optuna-optimized CNN architecture
- **05-training-optimization**: Optimized training hyperparameters
- **06-augmented-CNN**: CNN trained with data augmentation
- **07-resnet50**: Transfer learning with ResNet50

## Notebook setup

### Imports

In [1]:
# Standard library imports
import json
import pickle
import time

# Third party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from torchvision import datasets, transforms

# Package imports
import image_classification_tools.pytorch.plotting as plots

# Local imports
import configuration as config


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.10/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.10/dist-packages/traitlets/config/application.py", line 1043, in launch_instance
    app.start()
  File "/usr/local/lib/python3.10/dist-packages/ipykernel/kernelapp.p

AttributeError: _ARRAY_API not found

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 1. Load model metadata

In [ ]:
# Load model configurations from JSON
with open(config.MODELS_DIR / 'models_config.json', 'r') as f:
    models_metadata = json.load(f)['models']

print(f'Loaded metadata for {len(models_metadata)} models')

### Prepare test samples

In [ ]:
# Prepare test samples for cold start testing
grayscale_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
rgb_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

grayscale_dataset = datasets.CIFAR10(root=config.DATA_DIR, train=False, transform=grayscale_transform)
rgb_dataset = datasets.CIFAR10(root=config.DATA_DIR, train=False, transform=rgb_transform)

grayscale_sample, _ = grayscale_dataset[0]
rgb_sample, _ = rgb_dataset[0]

## 2. Measure cold start times

In [ ]:
cold_start_times = {}

for model_info in models_metadata:

    model_name = model_info['name']
    model_path = config.MODELS_DIR / model_info['model_file']
    
    if not model_path.exists():
        print(f'Skipping {model_name} - model file not found')
        continue
    
    # Select appropriate test sample
    sample = grayscale_sample if model_info['input_type'] == 'grayscale' else rgb_sample
    sample_batch = sample.unsqueeze(0)
    
    # Measure cold start time (average over 5 runs)
    times = []

    for _ in range(5):

        start_time = time.time()
        
        # Load model from disk (weights_only=False for models saved with torch.save(model))
        loaded = torch.load(model_path, map_location=config.DEVICE, weights_only=False)
        
        # Check if it's a state dict or full model
        if isinstance(loaded, dict):
            print(f'Skipping {model_name} - saved as state_dict (not full model)')
            break
        
        model = loaded
        model.to(config.DEVICE)
        model.eval()
        
        # Make single prediction
        with torch.no_grad():
            _ = model(sample_batch.to(config.DEVICE))
        
        times.append(time.time() - start_time)
        
        # Clean up
        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # Only record if we successfully measured
    if times:
        cold_start_times[model_name] = np.mean(times)
        print(f'{model_name}: {cold_start_times[model_name]*1000:.2f} ms')

print(f'\nCold start measurement complete for {len(cold_start_times)} models')

## 3. Measure average inference rate

In [ ]:
inference_rates = {}
n_samples = 1000

for model_info in models_metadata:

    model_name = model_info['name']
    model_path = config.MODELS_DIR / model_info['model_file']
    
    if not model_path.exists():
        print(f'Skipping {model_name} - model file not found')
        continue
    
    # Load appropriate dataset
    dataset = grayscale_dataset if model_info['input_type'] == 'grayscale' else rgb_dataset
    
    # Load model
    loaded = torch.load(model_path, map_location=config.DEVICE, weights_only=False)
    
    # Check if it's a state dict or full model
    if isinstance(loaded, dict):
        print(f'Skipping {model_name} - saved as state_dict (not full model)')
        continue
    
    model = loaded
    model.to(config.DEVICE)
    model.eval()
    
    # Prepare batch of samples
    samples = []

    for i in range(n_samples):
        img, _ = dataset[i]
        samples.append(img)
    
    batch = torch.stack(samples).to(config.DEVICE)
    
    # Warm up
    with torch.no_grad():
        _ = model(batch[:10])
    
    # Measure inference time
    start_time = time.time()
    
    with torch.no_grad():
        _ = model(batch)
    
    elapsed_time = time.time() - start_time
    
    # Calculate images per second
    inference_rate = n_samples / elapsed_time
    inference_rates[model_name] = inference_rate
    
    print(f'{model_name}: {inference_rate:.2f} images/sec ({elapsed_time*1000/n_samples:.2f} ms/image)')
    
    # Clean up
    del model, batch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\nInference rate measurement complete for {len(inference_rates)} models')

## 4. Load performance results from disk

In [ ]:
results = []

for model_info in models_metadata:

    model_name = model_info['name']
    results_path = config.RESULTS_DIR / model_info['results_file']
    
    if not results_path.exists():
        print(f"Skipping {model_name} - results file not found")
        continue
    
    # Load saved test results
    with open(results_path, 'rb') as f:
        data = pickle.load(f)
    
    # Calculate model size (4 bytes per float32 parameter)
    model_size_mb = (data['total_params'] * 4) / (1024 ** 2)
    
    # Compile results
    results.append({
        'Model': model_name,
        'Description': model_info['description'],
        'Total parameters': data['total_params'],
        'Trainable parameters': data['trainable_params'],
        'Size (MB)': model_size_mb,
        'Test accuracy (%)': data['test_accuracy'],
        'Cold start (ms)': cold_start_times.get(model_name, 0) * 1000,
        'Inference rate (img/s)': inference_rates.get(model_name, 0),
        'true_labels': np.array(data['true_labels']),
        'predictions': np.array(data['predictions']),
        'all_probs': np.array(data['all_probs'])
    })

print(f'Loaded results for {len(results)} models')

## 5. Results summary

In [ ]:
# Create summary dataframe
summary_df = pd.DataFrame([
    {
        'Model': r['Model'],
        'Description': r['Description'],
        'Parameters': f"{r['Total parameters']:,}",
        'Size (MB)': f"{r['Size (MB)']:.2f}",
        'Accuracy (%)': f"{r['Test accuracy (%)']:.2f}",
        'Cold start (ms)': f"{r['Cold start (ms)']:.2f}",
        'Inference (img/s)': f"{r['Inference rate (img/s)']:.2f}"
    }
    for r in results
])

summary_df

## 6. Performance visualizations

### Model performance metrics

In [ ]:
# Create a 2x2 subplot figure with all performance metrics
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 10))

# 1. Model performance comparison (Accuracy)
ax1.set_title('Model performance comparison', fontweight='bold', fontsize=14, pad=15)
sorted_results = sorted(results, key=lambda x: x['Test accuracy (%)'])
models_acc = [r['Model'] for r in sorted_results]
accuracies = [r['Test accuracy (%)'] for r in sorted_results]
bars1 = ax1.barh(models_acc, accuracies, color='black')

for i, (bar, acc) in enumerate(zip(bars1, accuracies)):
    ax1.text(acc + 0.5, i, f'{acc:.1f}%', va='center')

ax1.set_xlabel('Test accuracy (%)')
ax1.set_xlim(0, 100)

# 2. Model efficiency comparison
ax2.set_title('Model efficiency comparison', fontweight='bold', fontsize=14, pad=15)

efficiency_results = []

for r in results:
    params_millions = r['Total parameters'] / 1e6
    efficiency = r['Test accuracy (%)'] / params_millions
    efficiency_results.append({
        'Model': r['Model'],
        'Accuracy/parameter (M)': efficiency
    })

efficiency_results = sorted(
    efficiency_results,
    key=lambda x: x['Accuracy/parameter (M)'],
    reverse=True
)

models_eff = [r['Model'] for r in efficiency_results]
efficiencies = [r['Accuracy/parameter (M)'] for r in efficiency_results]
bars2 = ax2.barh(models_eff, efficiencies, color='black')

for i, (bar, eff) in enumerate(zip(bars2, efficiencies)):
    ax2.text(eff + 0.5, i, f'{eff:.1f}', va='center')

ax2.set_xscale('log')
ax2.set_xlabel('Accuracy per million parameters')

# 3. Inference rate comparison
ax3.set_title('Inference rate comparison', fontweight='bold', fontsize=14, pad=15)
sorted_by_inference = sorted(results, key=lambda x: x['Inference rate (img/s)'])
models_inf = [r['Model'] for r in sorted_by_inference]
inference_rates_list = [r['Inference rate (img/s)'] for r in sorted_by_inference]
bars3 = ax3.barh(models_inf, inference_rates_list, color='black')

for i, (bar, rate) in enumerate(zip(bars3, inference_rates_list)):
    ax3.text(rate + 50, i, f'{rate:.0f}', va='center')

ax3.set_xlabel('Inference rate (images/second)')

# 4. Cold start time comparison
ax4.set_title('Cold start time comparison', fontweight='bold', fontsize=14, pad=15)
sorted_by_cold_start = sorted(results, key=lambda x: x['Cold start (ms)'])
models_cold = [r['Model'] for r in sorted_by_cold_start]
cold_start_list = [r['Cold start (ms)'] for r in sorted_by_cold_start]
bars4 = ax4.barh(models_cold, cold_start_list, color='black')

for i, (bar, time_ms) in enumerate(zip(bars4, cold_start_list)):
    ax4.text(time_ms + 1, i, f'{time_ms:.1f}', va='center')
ax4.set_xlabel('Cold start time (milliseconds)')

plt.tight_layout()
plt.show()

## 7. Per-class performance analysis

In [ ]:
# Calculate per-class accuracy for each model
class_accuracies = {}

for r in results:
    true = r['true_labels']
    pred = r['predictions']
    
    # Calculate accuracy for each class
    class_acc = []

    for i in range(10):
        mask = (true == i)

        if mask.sum() > 0:
            acc = (pred[mask] == i).sum() / mask.sum() * 100
            class_acc.append(acc)

        else:
            class_acc.append(0)
    
    class_accuracies[r['Model']] = class_acc

# Create dataframe
class_acc_df = pd.DataFrame(class_accuracies, index=config.CLASS_NAMES)

In [ ]:
# Create heatmap
fig, ax = plt.subplots(figsize=(8, 5))

ax.set_title('Per-class accuracy comparison', fontweight='bold', fontsize=14, pad=15)

im = ax.imshow(class_acc_df.T, cmap='viridis', aspect='auto', vmin=0, vmax=100)

# Set ticks and labels
ax.set_xticks(np.arange(len(config.CLASS_NAMES)))
ax.set_yticks(np.arange(len(results)))
ax.set_xticklabels(config.CLASS_NAMES, rotation=45, ha='right')
ax.set_yticklabels([r['Model'] for r in results])

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Accuracy (%)', rotation=270, labelpad=20)

# Add text annotations
for i in range(len(results)):
    for j in range(len(config.CLASS_NAMES)):
        ax.text(j, i, f'{class_acc_df.iloc[j, i]:.1f}',
                ha='center', va='center', color='black')

ax.set_xlabel('CIFAR-10 class')
ax.set_ylabel('Model')

plt.tight_layout()
plt.show()

## 8. Confusion matrix comparison

In [ ]:
# Find best and worst models by accuracy
sorted_by_acc = sorted(results, key=lambda x: x['Test accuracy (%)'])
worst_model = sorted_by_acc[0]
best_model = sorted_by_acc[-1]

# Plot confusion matrix for worst model
print(f"Worst model: {worst_model['Model']} - Accuracy: {worst_model['Test accuracy (%)']:.2f}%")

fig1, ax1 = plots.plot_confusion_matrix(
    worst_model['true_labels'], 
    worst_model['predictions'], 
    config.CLASS_NAMES,
    figsize=(8, 8)
)

ax1.set_title(
    f'{worst_model["Model"]} - Accuracy: {worst_model["Test accuracy (%)"]:.1f}%', 
    fontsize=12, fontweight='bold', pad=15
)

plt.show()

# Plot confusion matrix for best model
print(f"\nBest model: {best_model['Model']} - Accuracy: {best_model['Test accuracy (%)']:.2f}%")

fig2, ax2 = plots.plot_confusion_matrix(
    best_model['true_labels'], 
    best_model['predictions'], 
    config.CLASS_NAMES,
    figsize=(8, 8)
)

ax2.set_title(
    f'{best_model["Model"]} - Accuracy: {best_model["Test accuracy (%)"]:.1f}%', 
    fontsize=12, fontweight='bold', pad=15
)

plt.show()
